# PHASE1: Chargement,Exploration, Nettoyage et Préparation des Données

Cette phase permet de charger vos trois fichiers CSV, de nettoyer les structures de données complexes (les listes stockées sous forme de chaînes de caractères) et de restructurer le tout pour créer l'historique de visionnage au format demandé.

In [25]:
import pandas as pd
import ast
import json


#Chargement des données

user=pd.read_csv("./data/user_profiles_dataset.csv")
ratings=pd.read_csv("./data/ratings.csv")
movies=pd.read_csv("./data/movies.csv")


In [13]:

#Explorations

print(f"Catalogue Films : {movies.shape[0]} lignes, {movies.shape[1]} colonnes")
print(f"Notes Utilisateurs : {ratings.shape[0]} lignes, {ratings.shape[1]} colonnes")
print(f"Profils Démographiques : {user.shape[0]} lignes, {user.shape[1]} colonnes\n")



Catalogue Films : 62423 lignes, 3 colonnes
Notes Utilisateurs : 1048575 lignes, 4 colonnes
Profils Démographiques : 30 lignes, 10 colonnes



In [ ]:
#Nettoyage

def extraire_liste(valeur):
    if pd.isna(valeur):
        return []
    try:
        return ast.literal_eval(valeur)
    except (ValueError, SyntaxError):
        return []

user_clean = user.copy()
user_clean['interests'] = user_clean['interests'].apply(extraire_liste)
user_clean['activity_log'] = user_clean['activity_log'].apply(extraire_liste)



Profils nettoyés avec succès. Exemple d'intérêts pour le premier utilisateur :
['cinema', 'music']


In [28]:
#Fusion des données

historique_fusionne = pd.merge(ratings,movies, on='movieId', how='inner')

dictionnaire_historiques = {}
groupes_par_utilisateur = historique_fusionne.groupby('userId')

for identifiant_utilisateur, groupe in groupes_par_utilisateur:
    liste_films_vus = []
    for _, ligne in groupe.iterrows():
        liste_films_vus.append({
            "film": ligne['title'],
            "genre": ligne['genres'].split('|') if pd.notna(ligne['genres']) else [],
            "note": float(ligne['rating']),
            
        })
    dictionnaire_historiques[identifiant_utilisateur] = liste_films_vus

print(f"Historique de visionnage reconstruit pour {len(dictionnaire_historiques)} utilisateurs issus des notations.")

Historique de visionnage reconstruit pour 7045 utilisateurs issus des notations.


In [29]:
#Creation des profils utilisateurs

utilisateurs_structures = {}

for _, ligne in user_clean.iterrows():
    id_utilisateur = ligne['id']
    
    genres_preferes = [interet for interet in ligne['interests'] if interet not in ['cinema', 'music']]
    if not genres_preferes:
        genres_preferes = ["action", "drama"] 
        
    historique_utilisateur = dictionnaire_historiques.get(id_utilisateur, [])
    
    utilisateurs_structures[id_utilisateur] = {
        "name": ligne['name'],
        "age": int(ligne['age']),
        "preferences": genres_preferes,
        "watch_history": historique_utilisateur
    }

id_test = list(utilisateurs_structures.keys())[0]
print(json.dumps(utilisateurs_structures[id_test], indent=4, ensure_ascii=False)[:650] + "\n...")

{
    "name": "John Doe",
    "age": 23,
    "preferences": [
        "action",
        "drama"
    ],
    "watch_history": [
        {
            "film": "Pulp Fiction (1994)",
            "genre": [
                "Comedy",
                "Crime",
                "Drama",
                "Thriller"
            ],
            "note": 5.0
        },
        {
            "film": "Three Colors: Red (Trois couleurs: Rouge) (1994)",
            "genre": [
                "Drama"
            ],
            "note": 3.5
        },
        {
            "film": "Three Colors: Blue (Trois couleurs: Bleu) (1993)",
            "genre": [
           
...
